In [ ]:
#импорты и пути
from pathlib import Path
import os
import math
import time

import duckdb
import pandas as pd

if (Path("processed") / "mart_events.parquet").exists():
    DATA_DIR = Path(".")

elif (Path("data") / "processed" / "mart_events.parquet").exists():
    DATA_DIR = Path("data")

else:
    raise FileNotFoundError(
        "Не найден mart_events.parquet. "
        "Запусти ноутбук из корня проекта или из папки data."
    )


PROCESSED_DIR = DATA_DIR / "processed"

CACHE_DIR = DATA_DIR / "mart_cache" / "mart_questions"
TEMP_DIR = DATA_DIR / "tmp_duckdb"


MART_EVENTS_PATH = PROCESSED_DIR / "mart_events.parquet"
QUESTIONS_PATH = PROCESSED_DIR / "questions_clean.parquet"

MART_QUESTIONS_PATH = PROCESSED_DIR / "mart_questions.parquet"


# технические файлы. в итоговую витрину они не попадают.

BUNDLE_ELAPSED_PATH = (
    CACHE_DIR / "question_bundle_elapsed.parquet"
)

BASE_METRICS_PATH = (
    CACHE_DIR / "question_metrics_base.parquet"
)


CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for path in [
    MART_EVENTS_PATH,
    QUESTIONS_PATH,
]:
    assert path.exists(), (
        f"Не найден файл: {path.resolve()}"
    )


def sql_path(path: Path) -> str:
    return (
        str(path.resolve())
        .replace("\\", "/")
        .replace("'", "''")
    )


MART_EVENTS = sql_path(
    MART_EVENTS_PATH
)

QUESTIONS = sql_path(
    QUESTIONS_PATH
)

MART_QUESTIONS = sql_path(
    MART_QUESTIONS_PATH
)

BUNDLE_ELAPSED = sql_path(
    BUNDLE_ELAPSED_PATH
)

BASE_METRICS = sql_path(
    BASE_METRICS_PATH
)

TEMP = sql_path(
    TEMP_DIR
)


cpu_count = os.cpu_count() or 4

DUCKDB_THREADS = min(
    8,
    max(4, cpu_count)
)


try:
    import psutil

    total_ram_gb = (
        psutil.virtual_memory().total
        / 1024**3
    )

    memory_gb = max(
        2,
        min(
            8,
            int(total_ram_gb * 0.55)
        )
    )

except Exception:
    total_ram_gb = None
    memory_gb = 4


DUCKDB_MEMORY_LIMIT = (
    f"{memory_gb}GB"
)


con = duckdb.connect()

con.execute(
    f"SET threads = {DUCKDB_THREADS}"
)

con.execute(
    f"SET memory_limit = "
    f"'{DUCKDB_MEMORY_LIMIT}'"
)

con.execute(
    f"SET temp_directory = '{TEMP}'"
)

con.execute(
    "SET preserve_insertion_order = false"
)


print(
    "DuckDB:",
    duckdb.__version__
)

print(
    "Threads:",
    DUCKDB_THREADS
)

print(
    "Memory limit:",
    DUCKDB_MEMORY_LIMIT
)

if total_ram_gb is not None:
    print(
        f"RAM компьютера: "
        f"{total_ram_gb:.1f} GB"
    )

print(
    "mart_events:",
    MART_EVENTS_PATH.resolve()
)

print(
    "questions:",
    QUESTIONS_PATH.resolve()
)

print(
    "result:",
    MART_QUESTIONS_PATH.resolve()
)

DuckDB: 1.5.5
Threads: 8
Memory limit: 8GB
RAM компьютера: 16.0 GB
mart_events: /Users/anna/проект шар/data/processed/mart_events.parquet
questions: /Users/anna/проект шар/data/processed/questions_clean.parquet
result: /Users/anna/проект шар/data/processed/mart_questions.parquet


In [40]:
#границы сегментов
BEGINNER_MIN_QUESTION = 1
BEGINNER_MAX_QUESTION = 20

EXPERIENCED_MIN_QUESTION = 101


print(
    "Beginner:",
    f"{BEGINNER_MIN_QUESTION}"
    f"–{BEGINNER_MAX_QUESTION}"
)

print(
    "Experienced:",
    f"{EXPERIENCED_MIN_QUESTION}+"
)

Beginner: 1–20
Experienced: 101+


In [ ]:
# статистика mart_events
events_dq = con.sql(f"""
SELECT

    COUNT(*) AS events,

    COUNT(*) FILTER (
        WHERE content_type_id = 0
    ) AS question_events,

    COUNT(DISTINCT user_id)
        AS users,

    COUNT(DISTINCT question_id)
        FILTER (
            WHERE content_type_id = 0
        ) AS questions_seen

FROM read_parquet('{MART_EVENTS}')
""").df()


display(events_dq)

,events,question_events,users,questions_seen
0,101230332,99271300,393656,13523


In [ ]:
# проверка времени внутри question bundle
bundle_dq = con.sql(f"""
WITH bundles AS (

    SELECT
        user_id,
        task_container_id,

        COUNT(
            DISTINCT
            prior_question_elapsed_time
        ) FILTER (
            WHERE
                prior_question_elapsed_time
                IS NOT NULL
        ) AS elapsed_variants

    FROM read_parquet(
        '{MART_EVENTS}'
    )

    WHERE content_type_id = 0

    GROUP BY
        user_id,
        task_container_id
)

SELECT

    COUNT(*) AS bundles,

    COUNT(*) FILTER (
        WHERE elapsed_variants > 1
    ) AS inconsistent_bundles

FROM bundles
""").df()


display(bundle_dq)


assert (
    int(
        bundle_dq.loc[
            0,
            "inconsistent_bundles"
        ]
    )
    == 0
), (
    "Внутри одного question bundle "
    "найдено несколько значений "
    "prior_question_elapsed_time"
)


print(
    "✅ elapsed time консистентен "
    "внутри question bundle"
)

,bundles,inconsistent_bundles
0,76483597,0


✅ elapsed time консистентен внутри question bundle


In [43]:
# восстанавливаем elapsed time текущего bundle
if BUNDLE_ELAPSED_PATH.exists():
    BUNDLE_ELAPSED_PATH.unlink()


started = time.perf_counter()


con.execute(f"""
COPY (

    WITH bundles AS (

        SELECT

            user_id,

            task_container_id,

            MIN(timestamp)
                AS bundle_timestamp,

            MIN(row_id)
                AS bundle_first_row_id,

            MIN(
                prior_question_elapsed_time
            ) AS prior_bundle_elapsed_time

        FROM read_parquet(
            '{MART_EVENTS}'
        )

        WHERE
            content_type_id = 0

        GROUP BY
            user_id,
            task_container_id
    ),

    shifted AS (

        SELECT

            user_id,

            task_container_id,

            LEAD(
                prior_bundle_elapsed_time
            ) OVER (

                PARTITION BY user_id

                ORDER BY
                    bundle_timestamp,
                    bundle_first_row_id

            ) AS elapsed_time

        FROM bundles
    )

    SELECT
        user_id,
        task_container_id,
        elapsed_time

    FROM shifted

)
TO '{BUNDLE_ELAPSED}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD,
    COMPRESSION_LEVEL 1,
    ROW_GROUP_SIZE 250000
)
""")


elapsed = (
    time.perf_counter()
    - started
)


print(
    "✅ Время ответа "
    "привязано к question bundle"
)

print(
    f"Время выполнения: "
    f"{elapsed / 60:.2f} min"
)

✅ Время ответа привязано к question bundle
Время выполнения: 0.10 min


In [44]:
# анализ распределения времени
elapsed_distribution = con.sql(f"""
SELECT

    COUNT(*) FILTER (
        WHERE elapsed_time > 0
    ) AS observations,

    approx_quantile(
        elapsed_time,
        0.25
    ) FILTER (
        WHERE elapsed_time > 0
    ) AS p25,

    approx_quantile(
        elapsed_time,
        0.50
    ) FILTER (
        WHERE elapsed_time > 0
    ) AS p50,

    approx_quantile(
        elapsed_time,
        0.75
    ) FILTER (
        WHERE elapsed_time > 0
    ) AS p75,

    approx_quantile(
        elapsed_time,
        0.95
    ) FILTER (
        WHERE elapsed_time > 0
    ) AS p95

FROM read_parquet(
    '{BUNDLE_ELAPSED}'
)
""").df()


display(elapsed_distribution)

,observations,p25,p50,p75,p95
0,75956294,15017.84034,19693.498126,27187.366105,53159.816645


In [53]:
FAST_THRESHOLD_MS = float(
    elapsed_distribution.loc[
        0,
        "p25"
    ]
)

SLOW_THRESHOLD_MS = float(
    elapsed_distribution.loc[
        0,
        "p75"
    ]
)


print(
    "порог быстрого ответаг:",
    f"{FAST_THRESHOLD_MS:,.0f} ms",
    f"= {FAST_THRESHOLD_MS / 1000:.1f} sec"
)

print(
    "порог медленного ответа:",
    f"{SLOW_THRESHOLD_MS:,.0f} ms",
    f"= {SLOW_THRESHOLD_MS / 1000:.1f} sec"
)

порог быстрого ответаг: 15,018 ms = 15.0 sec
порог медленного ответа: 27,187 ms = 27.2 sec


In [54]:
# метрики по каждому вопросу
if BASE_METRICS_PATH.exists():
    BASE_METRICS_PATH.unlink()


started = time.perf_counter()


con.execute(f"""
COPY (

    SELECT

        e.question_id,

        -- Основные метрики

        COUNT(*) AS attempts,


        COUNT(
            DISTINCT e.user_id
        ) AS users_count,


        SUM(
            CASE
                WHEN e.answered_correctly = 1
                THEN 1
                ELSE 0
            END
        ) AS correct_answers,


        SUM(
            CASE
                WHEN e.answered_correctly = 0
                THEN 1
                ELSE 0
            END
        ) AS incorrect_answers,


        AVG(
            CAST(
                e.answered_correctly
                AS DOUBLE
            )
        ) AS accuracy,


        -- Время ответа

        MEDIAN(
            b.elapsed_time
        ) FILTER (
            WHERE b.elapsed_time > 0
        ) AS median_elapsed_time,


        AVG(
            b.elapsed_time
        ) FILTER (
            WHERE b.elapsed_time > 0
        ) AS mean_elapsed_time,


        -- Beginner: 1–20 вопрос пользователя

        AVG(
            CAST(
                e.answered_correctly
                AS DOUBLE
            )
        ) FILTER (

            WHERE
                e.question_number
                BETWEEN
                    {BEGINNER_MIN_QUESTION}
                    AND
                    {BEGINNER_MAX_QUESTION}

        ) AS beginner_accuracy,


        -- Experienced: 101+ вопрос пользователя


        AVG(
            CAST(
                e.answered_correctly
                AS DOUBLE
            )
        ) FILTER (

            WHERE
                e.question_number
                >= {EXPERIENCED_MIN_QUESTION}

        ) AS experienced_accuracy,


        -- Доля всех ответов с известным временем,
        -- которые одновременно:
        -- 1. неправильные;
        -- 2. быстрые.


        SUM(
            CASE

                WHEN
                    e.answered_correctly = 0

                    AND b.elapsed_time > 0

                    AND b.elapsed_time
                        <= {FAST_THRESHOLD_MS}

                THEN 1

                ELSE 0

            END
        ) * 1.0

        /

        NULLIF(

            COUNT(*) FILTER (
                WHERE b.elapsed_time > 0
            ),

            0

        ) AS fast_incorrect_rate,


        -- Доля всех ответов с известным временем,
        -- которые одновременно:
        -- 1. неправильные;
        -- 2. медленные.

        SUM(
            CASE

                WHEN
                    e.answered_correctly = 0

                    AND b.elapsed_time
                        >= {SLOW_THRESHOLD_MS}

                THEN 1

                ELSE 0

            END
        ) * 1.0

        /

        NULLIF(

            COUNT(*) FILTER (
                WHERE b.elapsed_time > 0
            ),

            0

        ) AS slow_incorrect_rate,


        -- Средняя серия ошибок пользователя
        -- перед текущим вопросом

        AVG(
            CAST(
                e.error_streak
                AS DOUBLE
            )
        ) AS error_streak_before_avg


    FROM read_parquet(
        '{MART_EVENTS}'
    ) e


    LEFT JOIN read_parquet(
        '{BUNDLE_ELAPSED}'
    ) b

        ON
            e.user_id
            = b.user_id

        AND
            e.task_container_id
            = b.task_container_id


    WHERE

        e.content_type_id = 0

        AND e.question_id IS NOT NULL

        AND e.answered_correctly
            IN (0, 1)


    GROUP BY
        e.question_id

)
TO '{BASE_METRICS}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD,
    COMPRESSION_LEVEL 3,
    ROW_GROUP_SIZE 250000
)
""")


elapsed = (
    time.perf_counter()
    - started
)


print(
    "✅ Основные метрики вопросов рассчитаны"
)

print(
    f"Время выполнения: "
    f"{elapsed / 60:.2f} min"
)

✅ Основные метрики вопросов рассчитаны
Время выполнения: 0.27 min


In [47]:
# распределение attempts
attempts_distribution = con.sql(f"""
SELECT

    COUNT(*) AS questions,

    MIN(attempts)
        AS min_attempts,

    quantile_disc(
        attempts,
        0.01
    ) AS p01,

    quantile_disc(
        attempts,
        0.05
    ) AS p05,

    quantile_disc(
        attempts,
        0.10
    ) AS p10,

    quantile_disc(
        attempts,
        0.25
    ) AS p25,

    quantile_disc(
        attempts,
        0.50
    ) AS p50,

    quantile_disc(
        attempts,
        0.75
    ) AS p75,

    quantile_disc(
        attempts,
        0.90
    ) AS p90,

    quantile_disc(
        attempts,
        0.95
    ) AS p95,

    quantile_disc(
        attempts,
        0.99
    ) AS p99,

    MAX(attempts)
        AS max_attempts

FROM read_parquet(
    '{BASE_METRICS}'
)
""").df()


display(attempts_distribution)

,questions,min_attempts,p01,p05,p10,p25,p50,p75,p90,p95,p99,max_attempts
0,13523,1,76,153,283,1764,4732,8557,15950,22467,45977,213605


In [ ]:
# MIN_ATTEMPTS
p10_attempts = int(
    attempts_distribution.loc[
        0,
        "p10"
    ]
)


MIN_ATTEMPTS = max(
    100,
    p10_attempts
)


print(
    "P10 attempts:",
    f"{p10_attempts:,}"
)

print(
    "Зафиксированный MIN_ATTEMPTS:",
    f"{MIN_ATTEMPTS:,}"
)
#самые слабо наблюдаемые вопросы не участвуют в основном рейтинге сложности

P10 attempts: 283
Зафиксированный MIN_ATTEMPTS: 283


In [55]:
# создание витрины
if MART_QUESTIONS_PATH.exists():
    MART_QUESTIONS_PATH.unlink()


started = time.perf_counter()


con.execute(f"""
COPY (

    SELECT

        -- Поля вопроса

        q.question_id,

        q.bundle_id,

        q.part,

        q.tags,

        q.correct_answer,


        -- Основные метрики


        COALESCE(
            m.attempts,
            0
        ) AS attempts,


        COALESCE(
            m.users_count,
            0
        ) AS users_count,


        COALESCE(
            m.correct_answers,
            0
        ) AS correct_answers,


        COALESCE(
            m.incorrect_answers,
            0
        ) AS incorrect_answers,


        m.accuracy,


        CASE

            WHEN m.accuracy IS NULL
            THEN NULL

            ELSE
                1.0 - m.accuracy

        END AS difficulty,


        m.median_elapsed_time,


        m.mean_elapsed_time,


        -- Сложность на разных стадиях обучения

        m.beginner_accuracy,


        m.experienced_accuracy,


        m.experienced_accuracy
            - m.beginner_accuracy
            AS accuracy_gap,


        -- Надёжность оценки сложности

        CASE

            WHEN
                COALESCE(
                    m.attempts,
                    0
                ) >= {MIN_ATTEMPTS}

            THEN TRUE

            ELSE FALSE

        END AS enough_attempts,


        -- Дополнительные признаки из ТЗ

        m.fast_incorrect_rate,


        m.slow_incorrect_rate,


        m.error_streak_before_avg


    FROM read_parquet(
        '{QUESTIONS}'
    ) q


    LEFT JOIN read_parquet(
        '{BASE_METRICS}'
    ) m

        USING (question_id)


    ORDER BY
        q.question_id

)
TO '{MART_QUESTIONS}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD,
    COMPRESSION_LEVEL 3
)
""")


elapsed = (
    time.perf_counter()
    - started
)


print(
    "✅ mart_questions построена"
)

print(
    f"Время: {elapsed:.1f} sec"
)

print(
    "Файл:",
    MART_QUESTIONS_PATH.resolve()
)

print(
    "Размер:",
    f"{MART_QUESTIONS_PATH.stat().st_size / 1024**2:.2f} MB"
)

✅ mart_questions построена
Время: 0.0 sec
Файл: /Users/anna/проект шар/data/processed/mart_questions.parquet
Размер: 1.01 MB


In [50]:
# проверка схемы
EXPECTED_COLUMNS = [

    "question_id",
    "bundle_id",
    "part",
    "tags",
    "correct_answer",

    "attempts",
    "users_count",
    "correct_answers",
    "incorrect_answers",
    "accuracy",
    "difficulty",
    "median_elapsed_time",
    "mean_elapsed_time",

    "beginner_accuracy",
    "experienced_accuracy",
    "accuracy_gap",

    "enough_attempts",

    "fast_incorrect_rate",
    "slow_incorrect_rate",
    "error_streak_before_avg",
]


actual_columns = con.sql(f"""
DESCRIBE

SELECT *

FROM read_parquet(
    '{MART_QUESTIONS}'
)
""").df()["column_name"].tolist()


assert (
    actual_columns
    == EXPECTED_COLUMNS
), (
    "Схема mart_questions "
    "не соответствует ТЗ.\n\n"

    f"Ожидалось:\n"
    f"{EXPECTED_COLUMNS}\n\n"

    f"Получено:\n"
    f"{actual_columns}"
)


print(
    f"✅ В витрине ровно "
    f"{len(actual_columns)} полей"
)

print(
    "✅ Лишних полей нет"
)

✅ В витрине ровно 20 полей
✅ Лишних полей нет


In [56]:
# основные dq-проверки
dq = con.sql(f"""
SELECT

    COUNT(*)
        AS rows_total,


    COUNT(
        DISTINCT question_id
    ) AS unique_question_ids,


    -- accuracy ∈ [0, 1]

    COUNT(*) FILTER (

        WHERE
            accuracy IS NOT NULL

            AND NOT (
                accuracy
                BETWEEN 0 AND 1
            )

    ) AS bad_accuracy,


    -- difficulty = 1 - accuracy

    COUNT(*) FILTER (

        WHERE
            accuracy IS NOT NULL

            AND (

                difficulty IS NULL

                OR ABS(
                    difficulty
                    - (
                        1.0
                        - accuracy
                    )
                ) > 1e-12

            )

    ) AS bad_difficulty,


    -- correct + incorrect = attempts

    COUNT(*) FILTER (

        WHERE
            correct_answers
            + incorrect_answers
            <> attempts

    ) AS bad_answer_counts,


    -- enough_attempts

    COUNT(*) FILTER (

        WHERE
            enough_attempts
            <> (
                attempts
                >= {MIN_ATTEMPTS}
            )

    ) AS bad_enough_attempts,


    -- beginner_accuracy ∈ [0,1]


    COUNT(*) FILTER (

        WHERE
            beginner_accuracy
            IS NOT NULL

            AND NOT (
                beginner_accuracy
                BETWEEN 0 AND 1
            )

    ) AS bad_beginner_accuracy,


    -- experienced_accuracy ∈ [0,1]


    COUNT(*) FILTER (

        WHERE
            experienced_accuracy
            IS NOT NULL

            AND NOT (
                experienced_accuracy
                BETWEEN 0 AND 1
            )

    ) AS bad_experienced_accuracy,



    -- fast incorrect rate ∈ [0,1]


    COUNT(*) FILTER (

        WHERE
            fast_incorrect_rate
            IS NOT NULL

            AND NOT (
                fast_incorrect_rate
                BETWEEN 0 AND 1
            )

    ) AS bad_fast_incorrect_rate,


    -- slow incorrect rate ∈ [0,1]

    COUNT(*) FILTER (

        WHERE
            slow_incorrect_rate
            IS NOT NULL

            AND NOT (
                slow_incorrect_rate
                BETWEEN 0 AND 1
            )

    ) AS bad_slow_incorrect_rate


FROM read_parquet(
    '{MART_QUESTIONS}'
)
""").df()


display(dq)

,rows_total,unique_question_ids,bad_accuracy,bad_difficulty,bad_answer_counts,bad_enough_attempts,bad_beginner_accuracy,bad_experienced_accuracy,bad_fast_incorrect_rate,bad_slow_incorrect_rate
0,13523,13523,0,0,0,0,0,0,0,0


In [ ]:
r = dq.iloc[0]


#Одна строка = один вопрос


assert (
    int(r["rows_total"])
    == int(r["unique_question_ids"])
), (
    "В mart_questions есть "
    "дубли question_id"
)

# JOIN не создал дубликаты


source_questions_count = con.sql(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{QUESTIONS}'
)
""").fetchone()[0]


assert (
    int(r["rows_total"])
    == int(source_questions_count)
), (
    "JOIN с questions_clean "
    "изменил количество вопросов"
)


# Accuracy

assert (
    int(r["bad_accuracy"])
    == 0
), "Есть accuracy вне диапазона [0, 1]"


# Difficulty


assert (
    int(r["bad_difficulty"])
    == 0
), "Нарушено difficulty = 1 - accuracy"


# Correct + incorrect = attempts


assert (
    int(r["bad_answer_counts"])
    == 0
), (
    "correct_answers + incorrect_answers "
    "не равно attempts"
)

# enough_attempts


assert (
    int(r["bad_enough_attempts"])
    == 0
), "Некорректно рассчитан enough_attempts"


# Segment accuracy


assert (
    int(r["bad_beginner_accuracy"])
    == 0
)

assert (
    int(r["bad_experienced_accuracy"])
    == 0
)


# Additional rates


assert (
    int(r["bad_fast_incorrect_rate"])
    == 0
)

assert (
    int(r["bad_slow_incorrect_rate"])
    == 0
)


print(
    "✅ Одна строка = один question_id"
)

print(
    "✅ JOIN не создаёт дубликаты"
)

print(
    "✅ accuracy находится в [0, 1]"
)

print(
    "✅ difficulty = 1 - accuracy"
)

print(
    "✅ correct_answers + "
    "incorrect_answers = attempts"
)

print(
    "✅ enough_attempts рассчитан корректно"
)

print(
    "✅ beginner_accuracy находится в [0, 1]"
)

print(
    "✅ experienced_accuracy находится в [0, 1]"
)

print(
    "✅ fast_incorrect_rate находится в [0, 1]"
)

print(
    "✅ slow_incorrect_rate находится в [0, 1]"
)

✅ Одна строка = один question_id
✅ JOIN не создаёт дубликаты
✅ accuracy находится в [0, 1]
✅ difficulty = 1 - accuracy
✅ correct_answers + incorrect_answers = attempts
✅ enough_attempts рассчитан корректно
✅ beginner_accuracy находится в [0, 1]
✅ experienced_accuracy находится в [0, 1]
✅ fast_incorrect_rate находится в [0, 1]
✅ slow_incorrect_rate находится в [0, 1]


In [ ]:
# Надёжность сравнения beginner и experienced, поля не добавляются в итоговую витрину

MIN_SEGMENT_ATTEMPTS = 30

segment_support = con.sql(f"""
WITH support AS (

    SELECT
        question_id,

        COUNT(*) FILTER (
            WHERE question_number
                BETWEEN {BEGINNER_MIN_QUESTION}
                AND {BEGINNER_MAX_QUESTION}
        ) AS beginner_attempts,

        COUNT(*) FILTER (
            WHERE question_number
                >= {EXPERIENCED_MIN_QUESTION}
        ) AS experienced_attempts

    FROM read_parquet('{MART_EVENTS}')

    WHERE
        content_type_id = 0
        AND question_id IS NOT NULL
        AND answered_correctly IN (0, 1)

    GROUP BY question_id
)

SELECT
    q.question_id,
    q.beginner_accuracy,
    q.experienced_accuracy,
    q.accuracy_gap,

    COALESCE(s.beginner_attempts, 0)
        AS beginner_attempts,

    COALESCE(s.experienced_attempts, 0)
        AS experienced_attempts,

    (
        COALESCE(s.beginner_attempts, 0)
            >= {MIN_SEGMENT_ATTEMPTS}

        AND

        COALESCE(s.experienced_attempts, 0)
            >= {MIN_SEGMENT_ATTEMPTS}
    ) AS reliable_accuracy_gap

FROM read_parquet('{MART_QUESTIONS}') q

LEFT JOIN support s
    USING (question_id)

ORDER BY ABS(q.accuracy_gap) DESC NULLS LAST
""").df()

display(segment_support.head(20))

,question_id,beginner_accuracy,experienced_accuracy,accuracy_gap,beginner_attempts,experienced_attempts,reliable_accuracy_gap
0,11854,0.0,0.952598,0.952598,1,1097,False
1,137,0.0,0.943038,0.943038,1,790,False
2,12925,0.0,0.933775,0.933775,1,151,False
3,11726,0.0,0.927729,0.927729,1,678,False
4,12814,0.0,0.921986,0.921986,2,141,False
5,10786,0.0,0.918688,0.918688,1,2226,False
6,13200,0.0,0.915493,0.915493,1,142,False
7,11096,0.0,0.909265,0.909265,1,1047,False
8,11137,0.0,0.908751,0.908751,1,2137,False
9,12972,0.0,0.901786,0.901786,1,112,False


In [32]:
gap_reliability_summary = (
    segment_support[
        segment_support["accuracy_gap"].notna()
    ]
    .groupby(
        "reliable_accuracy_gap",
        dropna=False
    )
    .size()
    .reset_index(
        name="questions_count"
    )
)

display(gap_reliability_summary)

,reliable_accuracy_gap,questions_count
0,False,3878
1,True,8178


In [33]:
reliable_gap_top = (
    segment_support[
        segment_support["reliable_accuracy_gap"]
    ]
    .sort_values(
        "accuracy_gap",
        key=lambda s: s.abs(),
        ascending=False
    )
)

display(reliable_gap_top.head(20))

,question_id,beginner_accuracy,experienced_accuracy,accuracy_gap,beginner_attempts,experienced_attempts,reliable_accuracy_gap
164,4962,0.125984,0.656797,0.530813,127,1486,True
207,4438,0.195489,0.693382,0.497894,133,1360,True
212,8210,0.262500,0.755257,0.492757,160,1712,True
271,4833,0.265957,0.711749,0.445791,282,1464,True
273,8286,0.385542,0.829833,0.444291,166,1381,True
291,4624,0.225381,0.659790,0.434409,2427,7416,True
309,8998,0.424925,0.845686,0.420761,666,3778,True
311,4440,0.317536,0.737560,0.420024,211,1467,True
327,6307,0.282365,0.697585,0.415220,981,4140,True
330,4900,0.309353,0.723674,0.414322,139,1339,True


In [34]:
assert (
    segment_support.loc[
        segment_support["reliable_accuracy_gap"],
        "beginner_attempts"
    ] >= MIN_SEGMENT_ATTEMPTS
).all()

assert (
    segment_support.loc[
        segment_support["reliable_accuracy_gap"],
        "experienced_attempts"
    ] >= MIN_SEGMENT_ATTEMPTS
).all()

print(
    "✅ accuracy_gap считается надёжным только "
    "при достаточном числе попыток в обеих группах"
)

✅ accuracy_gap считается надёжным только при достаточном числе попыток в обеих группах


In [35]:
reliable_share = (
    segment_support[
        "accuracy_gap"
    ].notna()
    & segment_support[
        "reliable_accuracy_gap"
    ]
).sum() / segment_support[
    "accuracy_gap"
].notna().sum()

print(
    f"Надёжных accuracy_gap: "
    f"{reliable_share:.1%}"
)

Надёжных accuracy_gap: 67.8%


In [20]:
bad_accuracy_gap = con.sql(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{MART_QUESTIONS}'
)

WHERE

    beginner_accuracy IS NOT NULL

    AND experienced_accuracy IS NOT NULL

    AND (

        accuracy_gap IS NULL

        OR ABS(

            accuracy_gap

            - (
                experienced_accuracy
                - beginner_accuracy
            )

        ) > 1e-12

    )
""").fetchone()[0]


assert (
    bad_accuracy_gap
    == 0
), (
    "accuracy_gap рассчитан некорректно"
)


print(
    "✅ accuracy_gap = "
    "experienced_accuracy - beginner_accuracy"
)

✅ accuracy_gap = experienced_accuracy - beginner_accuracy


In [ ]:
# проверка основного рейтинга сложности
hardest_questions = con.sql(f"""
SELECT

    question_id,

    bundle_id,

    part,

    tags,

    attempts,

    users_count,

    correct_answers,

    incorrect_answers,

    accuracy,

    difficulty,

    median_elapsed_time,

    mean_elapsed_time,

    beginner_accuracy,

    experienced_accuracy,

    accuracy_gap,

    enough_attempts,

    fast_incorrect_rate,

    slow_incorrect_rate,

    error_streak_before_avg

FROM read_parquet(
    '{MART_QUESTIONS}'
)

WHERE
    enough_attempts = TRUE

ORDER BY
    difficulty DESC,
    attempts DESC

LIMIT 30
""").df()


display(hardest_questions)

,question_id,bundle_id,part,tags,attempts,users_count,correct_answers,incorrect_answers,accuracy,difficulty,median_elapsed_time,mean_elapsed_time,beginner_accuracy,experienced_accuracy,accuracy_gap,enough_attempts,fast_incorrect_rate,slow_incorrect_rate,error_streak_before_avg
0,10062,10062,6,8,7444,6523,683.0,6761.0,0.091752,0.908248,26500.0,28876.825805,0.020833,0.091509,0.070676,True,0.048147,0.446849,0.527405
1,7639,7638,7,118 42 21 162,8399,7725,845.0,7554.0,0.100607,0.899393,55000.0,58795.957431,0.030769,0.101230,0.070461,True,0.019962,0.861231,0.626384
2,3125,3123,4,157 12 92,10157,8616,1381.0,8776.0,0.135965,0.864035,24666.0,23993.546321,0.116883,0.134530,0.017647,True,0.164433,0.318439,0.703456
3,9220,9220,5,134,12359,10101,1789.0,10570.0,0.144753,0.855247,24000.0,28901.973577,0.089286,0.144552,0.055266,True,0.182597,0.356222,1.051946
4,7487,7484,7,42 18 160 35 122,9419,8779,1391.0,8028.0,0.147680,0.852320,70000.0,73207.347654,0.093023,0.149738,0.056714,True,0.031391,0.812728,0.500690
5,10924,10923,6,179,4080,3647,607.0,3473.0,0.148775,0.851225,27000.0,29121.768037,0.041667,0.150880,0.109213,True,0.062546,0.440286,0.444118
6,10095,10094,6,1,6132,5484,952.0,5180.0,0.155251,0.844749,26750.0,29546.508769,0.166667,0.156999,-0.009667,True,0.062613,0.425340,0.369374
7,2063,2063,3,136 92 29,176043,167814,28716.0,147327.0,0.163119,0.836881,25666.0,25591.828586,0.152688,0.244623,0.091936,True,0.105887,0.353623,0.785427
8,10061,10058,6,128,6780,5906,1151.0,5629.0,0.169764,0.830236,39250.0,41726.029434,0.166667,0.171317,0.004651,True,0.028393,0.704177,0.931711
9,6531,6529,6,89,6980,6129,1187.0,5793.0,0.170057,0.829943,32250.0,35211.413748,0.068966,0.170735,0.101770,True,0.030264,0.585819,0.382665


In [22]:
assert (
    len(hardest_questions) == 0

    or

    (
        hardest_questions["attempts"]
        >= MIN_ATTEMPTS
    ).all()
)


print(
    "✅ В рейтинг сложности входят "
    "только вопросы с enough_attempts = TRUE"
)

✅ В рейтинг сложности входят только вопросы с enough_attempts = TRUE


In [ ]:
# самые простые вопросы
easiest_questions = con.sql(f"""
SELECT

    question_id,

    bundle_id,

    part,

    attempts,

    users_count,

    accuracy,

    difficulty

FROM read_parquet(
    '{MART_QUESTIONS}'
)

WHERE
    enough_attempts = TRUE

ORDER BY
    difficulty ASC,
    attempts DESC

LIMIT 30
""").df()


display(easiest_questions)

,question_id,bundle_id,part,attempts,users_count,accuracy,difficulty
0,10626,10626,1,5719,5432,0.992656,0.007344
1,10440,10440,1,5732,5453,0.990579,0.009421
2,1679,1679,3,6948,6491,0.989349,0.010651
3,10451,10451,1,5844,5531,0.986653,0.013347
4,12265,12265,2,1448,1397,0.985497,0.014503
5,10400,10400,1,5575,5301,0.984753,0.015247
6,11215,11215,5,1154,1124,0.984402,0.015598
7,10500,10500,1,5505,5224,0.984378,0.015622
8,3381,3381,4,5819,5438,0.984018,0.015982
9,10425,10425,1,6084,5775,0.983892,0.016108


In [ ]:
# самые популярные вопросы
popular_questions = con.sql(f"""
SELECT

    question_id,

    bundle_id,

    part,

    attempts,

    users_count,

    accuracy,

    difficulty

FROM read_parquet(
    '{MART_QUESTIONS}'
)

ORDER BY
    attempts DESC

LIMIT 30
""").df()


display(popular_questions)

,question_id,bundle_id,part,attempts,users_count,accuracy,difficulty
0,6116,6116,5,213605,189998,0.279081,0.720919
1,6173,6173,5,202106,183727,0.295419,0.704581
2,4120,4120,5,199372,182313,0.275585,0.724415
3,175,175,1,195861,183151,0.359970,0.640030
4,7876,7876,1,190170,178772,0.418436,0.581564
5,7900,7900,1,180858,174141,0.826184,0.173816
6,2065,2063,3,176043,167814,0.633658,0.366342
7,2063,2063,3,176043,167814,0.163119,0.836881
8,2064,2063,3,176043,167814,0.636907,0.363093
9,4492,4492,5,173769,165149,0.455225,0.544775


In [ ]:
# вопросы с аномально большим временем ответа
long_time_threshold = con.sql(f"""
SELECT

    approx_quantile(
        median_elapsed_time,
        0.95
    )

FROM read_parquet(
    '{MART_QUESTIONS}'
)

WHERE

    enough_attempts = TRUE

    AND median_elapsed_time
        IS NOT NULL
""").fetchone()[0]


print(
    "P95 median elapsed time:",
    f"{long_time_threshold:,.0f} ms",
    f"= {long_time_threshold / 1000:.1f} sec"
)

P95 median elapsed time: 52,750 ms = 52.7 sec


In [26]:
slow_questions = con.sql(f"""
SELECT

    question_id,

    bundle_id,

    part,

    attempts,

    accuracy,

    difficulty,

    median_elapsed_time,

    mean_elapsed_time,

    slow_incorrect_rate

FROM read_parquet(
    '{MART_QUESTIONS}'
)

WHERE

    enough_attempts = TRUE

    AND median_elapsed_time
        >= {long_time_threshold}

ORDER BY
    median_elapsed_time DESC

LIMIT 30
""").df()


display(slow_questions)

,question_id,bundle_id,part,attempts,accuracy,difficulty,median_elapsed_time,mean_elapsed_time,slow_incorrect_rate
0,7260,7260,7,7926,0.750442,0.249558,89000.0,92210.929757,0.207024
1,7261,7260,7,7926,0.313525,0.686475,89000.0,92210.929757,0.639719
2,7262,7260,7,7924,0.247476,0.752524,89000.0,92213.564129,0.712443
3,7263,7260,7,7925,0.505363,0.494637,89000.0,92209.858219,0.457402
4,7264,7260,7,7926,0.584280,0.415720,89000.0,92210.929757,0.376756
5,11584,11580,7,2167,0.485464,0.514536,86000.0,89172.242523,0.464953
6,11583,11580,7,2167,0.600831,0.399169,86000.0,89172.242523,0.348131
7,11582,11580,7,2167,0.725427,0.274573,86000.0,89172.242523,0.224299
8,11581,11580,7,2167,0.379788,0.620212,86000.0,89172.242523,0.560748
9,11580,11580,7,2168,0.798432,0.201568,86000.0,89179.168146,0.165343


In [ ]:
# различия beginner / experienced
experience_gap = con.sql(f"""
SELECT

    question_id,

    bundle_id,

    part,

    attempts,

    accuracy,

    difficulty,

    beginner_accuracy,

    experienced_accuracy,

    accuracy_gap

FROM read_parquet(
    '{MART_QUESTIONS}'
)

WHERE

    enough_attempts = TRUE

    AND beginner_accuracy IS NOT NULL

    AND experienced_accuracy IS NOT NULL

ORDER BY
    ABS(accuracy_gap) DESC

LIMIT 30
""").df()


display(experience_gap)

,question_id,bundle_id,part,attempts,accuracy,difficulty,beginner_accuracy,experienced_accuracy,accuracy_gap
0,11854,11854,3,1128,0.950355,0.049645,0.0,0.952598,0.952598
1,137,137,1,978,0.933538,0.066462,0.0,0.943038,0.943038
2,11726,11724,3,703,0.926031,0.073969,0.0,0.927729,0.927729
3,10786,10783,6,2296,0.917683,0.082317,0.0,0.918688,0.918688
4,11096,11095,6,1082,0.907579,0.092421,0.0,0.909265,0.909265
5,11137,11135,6,2185,0.907551,0.092449,0.0,0.908751,0.908751
6,11126,11123,6,2533,0.901698,0.098302,0.0,0.901255,0.901255
7,11053,11051,6,2366,0.895182,0.104818,0.0,0.896791,0.896791
8,7554,7553,7,1313,0.894136,0.105864,0.0,0.894045,0.894045
9,12364,12363,3,306,0.882353,0.117647,0.0,0.886667,0.886667


In [ ]:
# потенциально проблемный контент
potentially_problematic = con.sql(f"""
SELECT

    question_id,

    bundle_id,

    part,

    tags,

    attempts,

    users_count,

    accuracy,

    difficulty,

    median_elapsed_time,

    mean_elapsed_time,

    beginner_accuracy,

    experienced_accuracy,

    accuracy_gap,

    fast_incorrect_rate,

    slow_incorrect_rate,

    error_streak_before_avg

FROM read_parquet(
    '{MART_QUESTIONS}'
)

WHERE
    enough_attempts = TRUE

ORDER BY

    slow_incorrect_rate DESC NULLS LAST,

    difficulty DESC NULLS LAST,

    median_elapsed_time DESC NULLS LAST

LIMIT 50
""").df()


display(potentially_problematic)

,question_id,bundle_id,part,tags,attempts,users_count,accuracy,difficulty,median_elapsed_time,mean_elapsed_time,beginner_accuracy,experienced_accuracy,accuracy_gap,fast_incorrect_rate,slow_incorrect_rate,error_streak_before_avg
0,7639,7638,7,118 42 21 162,8399,7725,0.100607,0.899393,55000.0,58795.957431,0.030769,0.101230,0.070461,0.019962,0.861231,0.626384
1,7487,7484,7,42 18 160 35 122,9419,8779,0.147680,0.852320,70000.0,73207.347654,0.093023,0.149738,0.056714,0.031391,0.812728,0.500690
2,11640,11640,7,37 153 21,1334,1270,0.197151,0.802849,62500.0,67191.121144,0.100000,0.201946,0.101946,0.020316,0.775771,0.614693
3,7396,7396,7,39 160 16 35 122,6853,6413,0.192325,0.807675,64200.0,66271.235953,0.204545,0.191099,-0.013447,0.028462,0.769945,0.719247
4,7402,7401,7,97 0 146 11 122,16938,15888,0.214901,0.785099,70600.0,72724.542606,0.185484,0.220110,0.034626,0.028524,0.742750,0.774176
5,7665,7664,7,97 160 16 35 122 162,5445,5177,0.222039,0.777961,55600.0,57502.837389,0.200000,0.223667,0.023667,0.030605,0.738016,0.296419
6,8161,8159,7,39 42 160 135 162,6869,6472,0.243704,0.756296,60400.0,62038.215721,0.193548,0.245065,0.051516,0.029330,0.717701,0.674916
7,7122,7121,7,145 37 21,3856,3572,0.224066,0.775934,55250.0,57618.290128,0.205882,0.224484,0.018602,0.046609,0.715371,0.711618
8,7262,7260,7,97 42 160 35 122,7924,7386,0.247476,0.752524,89000.0,92213.564129,0.111111,0.248802,0.137691,0.029509,0.712443,0.855250
9,7604,7603,7,97 37 21,6134,5704,0.194001,0.805999,42000.0,44906.060108,0.157895,0.194828,0.036933,0.039908,0.708655,0.363221


In [29]:
mart_preview = con.sql(f"""
SELECT *

FROM read_parquet(
    '{MART_QUESTIONS}'
)

ORDER BY
    question_id

LIMIT 20
""").df()


display(mart_preview)

,question_id,bundle_id,part,tags,correct_answer,attempts,users_count,correct_answers,incorrect_answers,accuracy,difficulty,median_elapsed_time,mean_elapsed_time,beginner_accuracy,experienced_accuracy,accuracy_gap,enough_attempts,fast_incorrect_rate,slow_incorrect_rate,error_streak_before_avg
0,0,0,1,51 131 162 38,0,6903,6380,6266.0,637.0,0.907721,0.092279,19000.0,19713.725490,0.841304,0.909759,0.068455,True,0.005664,0.010167,0.296538
1,1,1,1,131 36 81,1,7398,6829,6589.0,809.0,0.890646,0.109354,20000.0,19336.449864,0.830116,0.899701,0.069585,True,0.005149,0.010027,0.313463
2,2,2,1,131 101 162 92,0,44905,38322,24890.0,20015.0,0.554281,0.445719,23000.0,24602.229819,0.480648,0.581315,0.100667,True,0.005094,0.104051,0.539428
3,3,3,1,131 149 162 29,0,22973,20643,17906.0,5067.0,0.779437,0.220563,21000.0,21641.250710,0.670588,0.801865,0.131276,True,0.005764,0.028123,0.494319
4,4,4,1,131 5 162 38,3,31736,29006,19461.0,12275.0,0.613215,0.386785,21000.0,22087.282269,0.558334,0.671707,0.113374,True,0.016628,0.050675,0.499118
5,5,5,1,131 149 162 81,2,9727,8957,8383.0,1344.0,0.861828,0.138172,22000.0,22147.991761,0.799392,0.871884,0.072492,True,0.005252,0.019361,0.321271
6,6,6,1,10 94 162 92,2,56707,47199,26910.0,29797.0,0.474545,0.525455,23000.0,24036.320170,0.372232,0.522979,0.150747,True,0.022472,0.069807,0.683813
7,7,7,1,61 110 162 29,0,16585,15167,14363.0,2222.0,0.866024,0.133976,22000.0,21893.668439,0.802469,0.878093,0.075624,True,0.004894,0.022475,0.317697
8,8,8,1,131 13 162 92,3,8535,7895,7738.0,797.0,0.906620,0.093380,22000.0,22840.954508,0.816701,0.922296,0.105596,True,0.009404,0.025979,0.318453
9,9,9,1,10 164 81,3,47346,35147,14389.0,32957.0,0.303912,0.696088,23000.0,24171.051235,0.186511,0.328890,0.142379,True,0.010871,0.084294,0.590103


In [58]:
final_stats = con.sql(f"""
SELECT

    COUNT(*) AS questions,

    SUM(attempts)
        AS total_attempts,

    COUNT(*) FILTER (
        WHERE enough_attempts = TRUE
    ) AS questions_enough_attempts,

    AVG(accuracy)
        AS mean_question_accuracy,

    AVG(difficulty)
        AS mean_question_difficulty

FROM read_parquet(
    '{MART_QUESTIONS}'
)
""").df()


display(final_stats)


print()

print(
    "Файл:",
    MART_QUESTIONS_PATH.resolve()
)

print(
    "MIN_ATTEMPTS:",
    MIN_ATTEMPTS
)

print(
    "Beginner:",
    f"{BEGINNER_MIN_QUESTION}"
    f"–{BEGINNER_MAX_QUESTION}"
)

print(
    "Experienced:",
    f"{EXPERIENCED_MIN_QUESTION}+"
)

print(
    "Fast threshold:",
    f"{FAST_THRESHOLD_MS / 1000:.1f} sec"
)

print(
    "Slow threshold:",
    f"{SLOW_THRESHOLD_MS / 1000:.1f} sec"
)

print(
    "Количество полей:",
    len(EXPECTED_COLUMNS)
)

print()
print("✅ mart_questions.parquet сохранён")
print("✅ Одна строка = один вопрос")
print("✅ Все обязательные метрики рассчитаны")
print("✅ Лишних колонок нет")

,questions,total_attempts,questions_enough_attempts,mean_question_accuracy,mean_question_difficulty
0,13523,99271300.0,12171,0.70946,0.29054



Файл: /Users/anna/проект шар/data/processed/mart_questions.parquet
MIN_ATTEMPTS: 283
Beginner: 1–20
Experienced: 101+
Fast threshold: 15.0 sec
Slow threshold: 27.2 sec
Количество полей: 20

✅ mart_questions.parquet сохранён
✅ Одна строка = один вопрос
✅ Все обязательные метрики рассчитаны
✅ Лишних колонок нет
